In [0]:
gold_df = spark.table('github_stream_event_catalog.silver_layer.silver_github_events')
(gold_df.count())

# gold_df = spark.readStream.table(
#     "github_stream_event_catalog.silver_layer.silver_github_events"
# )


330

In [0]:
# gold_df.count()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

repo_activity = gold_df.groupBy('repo_name').agg(count('event_id').alias("No_Of_Events")).sort(col("No_Of_Events").desc())


# (repo_activity.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_repo_activity"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.repo_activity")
# )

repo_activity.show()

repo_activity.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("github_stream_event_catalog.gold_layer.repo_activity")


+--------------------+------------+
|           repo_name|No_Of_Events|
+--------------------+------------+
|tawfeeqa-source/H...|           3|
|SoliSpirit/v2ray-...|           3|
|   Tyndalllll/sglang|           2|
|pc-study/blog-images|           2|
|thepinkmile/Media...|           2|
|davidxjohnson/iot...|           2|
|TTMSADAO/ttmed-sa...|           2|
|extremelystiff/sa...|           1|
|andriilive/russia...|           1|
|kriezer12/FGAesth...|           1|
|AY2526S2-CS2103T-...|           1|
|aditya-jg/FiveM-M...|           1|
|thesameorg/splito...|           1|
|  yaseralnajjar/sifr|           1|
|   marvin7122/qlever|           1|
|CodeHub-Clients/C...|           1|
|DrXSolutions/Imag...|           1|
|lukabrand/LUCVOIC...|           1|
|penneyqafa/daily_...|           1|
|chvhouwe/previous...|           1|
+--------------------+------------+
only showing top 20 rows


In [0]:
actor_activity = gold_df.groupBy('actor_login').agg(count('event_id').alias("No_Of_Events")).sort(col("No_Of_Events").desc())


# (actor_activity.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_actor_activity"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.actor_activity")
# )
actor_activity.show()

actor_activity.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("github_stream_event_catalog.gold_layer.actor_activity")

+-------------------+------------+
|        actor_login|No_Of_Events|
+-------------------+------------+
|github-actions[bot]|          38|
|    dependabot[bot]|           6|
|         SoliSpirit|           4|
|      renovate[bot]|           3|
|    tawfeeqa-source|           3|
|         Tyndalllll|           2|
|           TTMSADAO|           2|
|         catalogbot|           2|
|          pull[bot]|           2|
|           pc-study|           2|
|          lukabrand|           1|
|          aditya-jg|           1|
|          garootman|           1|
|          kriezer12|           1|
|          darrenlhs|           1|
|     extremelystiff|           1|
|           chvhouwe|           1|
|         marvin7122|           1|
|         yodsanonDK|           1|
|    mayram01787-art|           1|
+-------------------+------------+
only showing top 20 rows


In [0]:
events_count = (
    gold_df
    .groupBy("type")
    .agg(count("event_id").alias("No_Of_Events"))
)

# calculate total events using window
result = events_count.withColumn(
    "Total_Events",
    sum("No_Of_Events").over(Window.partitionBy())
).withColumn(
    "percentage",
    round((col("No_Of_Events") / col("Total_Events")) * 100, 2)
)

# (result.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_event_type_distribution"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.event_type_distribution")
# )

result.show()

result.write \
.format("delta") \
.mode("overwrite") \
.option("mergeSchema", "true") \
.saveAsTable("github_stream_event_catalog.gold_layer.event_type_distribution")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------------+------------+----------+
|       type|No_Of_Events|Total_Events|percentage|
+-----------+------------+------------+----------+
|  PushEvent|         273|         330|     82.73|
|DeleteEvent|          16|         330|      4.85|
|CreateEvent|          41|         330|     12.42|
+-----------+------------+------------+----------+



In [0]:
activity_over_time = gold_df.groupBy("created_at").count().withColumnRenamed("count", "total_events")


# (activity_over_time.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_activity_over_time"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.activity_over_time")
# )
activity_over_time.show()

activity_over_time.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("github_stream_event_catalog.gold_layer.activity_over_time")

+-------------------+------------+
|         created_at|total_events|
+-------------------+------------+
|2026-03-09 09:07:10|          20|
|2026-03-09 09:08:29|          25|
|2026-03-09 09:08:40|          23|
|2026-03-09 09:08:39|           7|
|2026-03-09 09:09:01|           7|
|2026-03-09 09:09:02|          23|
|2026-03-09 09:07:22|          29|
|2026-03-09 09:07:33|          28|
|2026-03-09 09:07:44|          26|
|2026-03-09 09:08:07|          30|
|2026-03-09 09:08:18|          15|
|2026-03-09 09:07:55|          26|
|2026-03-09 09:08:17|          15|
|2026-03-09 09:08:28|           5|
|2026-03-09 09:07:43|           4|
|2026-03-09 09:08:50|          30|
|2026-03-09 09:07:54|           4|
|2026-03-09 09:07:32|           1|
|2026-03-09 09:07:21|           1|
|2026-03-09 09:07:09|          10|
+-------------------+------------+
only showing top 20 rows


In [0]:
branch_activity = gold_df.groupBy("branch").count().sort(col("count").desc())


# (branch_activity.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_branch_activity"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.branch_activity")
# )
branch_activity.show()

branch_activity.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("github_stream_event_catalog.gold_layer.branch_activity")

+--------------------+-----+
|              branch|count|
+--------------------+-----+
|                main|  217|
|              master|   21|
|              output|    4|
|             develop|    4|
|                 dev|    3|
|            gh-pages|    3|
|      release_v2.0.5|    1|
|   feature/treatment|    1|
|        beta-testing|    1|
|jordigomara/#0238...|    1|
|             patch-2|    1|
|   update/mill-1.1.3|    1|
|feat/feishu-docx-...|    1|
| renovate/tsdown-0.x|    1|
|dependabot/npm_an...|    1|
|renovate/major-gi...|    1|
|phase30-part24-lo...|    1|
|origon/Raphaelassets|    1|
|construct-batch-e...|    1|
|dependabot/github...|    1|
+--------------------+-----+
only showing top 20 rows


In [0]:
unique_developers = gold_df.select(
    countDistinct("actor_login").alias("unique_developers")
)


# (unique_developers.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_unique_developers"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.unique_developers")
# )
unique_developers.show()

unique_developers.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("github_stream_event_catalog.gold_layer.unique_developers")

+-----------------+
|unique_developers|
+-----------------+
|              276|
+-----------------+



In [0]:
org_activity = (
    gold_df
    .groupBy("org_login")
    .agg(count("event_id").alias("No_Of_Events"))
)

result = org_activity.withColumn(
    "Total_Events",
    sum("No_Of_Events").over(Window.partitionBy())
).withColumn(
    "percentage",
    round((col("No_Of_Events") / col("Total_Events")) * 100, 2)
)

result = result.sort(col('Total_Events').desc(), col('percentage').desc())
# (result.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_org_activity"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.org_activity")
# )
result.show()

result.write \
.format("delta") \
.mode("overwrite") \
.option("mergeSchema", "true") \
.saveAsTable("github_stream_event_catalog.gold_layer.org_activity")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------------+------------+------------+----------+
|           org_login|No_Of_Events|Total_Events|percentage|
+--------------------+------------+------------+----------+
|       personal_repo|         271|         330|     82.12|
|          giantswarm|           2|         330|      0.61|
|           VeloxChem|           1|         330|       0.3|
|           OpenShock|           1|         330|       0.3|
|         tigerbeetle|           1|         330|       0.3|
|       web-infra-dev|           1|         330|       0.3|
|           Curity-PS|           1|         330|       0.3|
|      hopr-framework|           1|         330|       0.3|
|             yaklang|           1|         330|       0.3|
|             giscube|           1|         330|       0.3|
|AY2526S2-CS2103T-...|           1|         330|       0.3|
|            Nomaryth|           1|         330|       0.3|
|            GeoOcean|           1|         330|       0.3|
|          bytechefhq|           1|     

In [0]:
gold_df.columns

['event_id',
 'type',
 'created_at',
 'public',
 'actor_id',
 'actor_login',
 'actor_display_login',
 'repo_id',
 'repo_name',
 'org_id',
 'org_login',
 'payload_push_id',
 'payload_ref',
 'payload_ref_type',
 'branch']

In [0]:
gold_df = gold_df.select('event_id', 'type', 'created_at', 'public', 'actor_id', 'actor_login', 'repo_name', 'repo_id','org_id', 'org_login', 'payload_ref_type', 'branch')


In [0]:
# (gold_df.writeStream
#  .format("delta")
#  .outputMode("complete")
#  .option(
#    "checkpointLocation",
#    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/gold_table"
#  )
#  .trigger(availableNow=True)
#  .table("github_stream_event_catalog.gold_layer.gold_table")
# )

In [0]:
%sql
-- create schema github_stream_event_catalog.gold_layer;

In [0]:
gold_df.write.mode('overwrite').saveAsTable('github_stream_event_catalog.gold_layer.gold_github_events')